# 拣金：在 RoboVAD 上跑 Robometer

给 RoboVAD 的 test（629 条）和 train（367 条）每条示范的左路视频算进度曲线，结果是一堆小 JSON（缓存）。

**怎么跑**
1. 运行时类型选 **T4 GPU**。
2. 把本机新打的 `dist/jianjin-0.1.0-py3-none-any.whl` 拖进 `/content/`（和上次的同名，但内容是新的，要重新拖）。
3. 先运行第 1 到 6 格（冒烟：只跑 3 条）。5a 下载权重有进度条；5b 加载约 5 分钟，期间显存是 0、内存在涨，正常。第 6 格会打印每条耗时和全量预计时间。
4. 没问题就运行第 7 格（全量，预计 2 到 3 小时）。中途断了：重开笔记本，重新拖 wheel，从第 1 格重跑，已经算完的会直接跳过。
5. 跑完运行第 8 格，会下载 `robometer_cache.zip`。放到项目的 `C:\Claude_code\jianjin\cache\` 下解压，告诉 Claude。

第 3 格会请求挂载你的 Google Drive，用来存缓存，断线也不丢。不想挂载就跳过第 3 格，缓存会放在 `/content/cache`（断线就没了）。


In [ ]:
# 1. GPU、装依赖（lerobot 锁 0.6.1：适配器用到了它的内部方法）
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q "lerobot[robometer]==0.6.1" psutil 2>&1 | tail -2
!pip install -q --force-reinstall --no-deps /content/jianjin-0.1.0-py3-none-any.whl 2>&1 | tail -2
import importlib.metadata as md
print("lerobot", md.version("lerobot"), "jianjin", md.version("jianjin"))

In [ ]:
# 2. 下载 RoboVAD（6.86 GB，带进度条，断了再运行会接着下），只解压左路视频和标注、关节数据
import os, urllib.request, zipfile
from tqdm.auto import tqdm
os.environ["HF_HUB_DISABLE_XET"] = "1"
URL = "https://zenodo.org/records/22754659/files/RoboVAD.zip?download=1"
ZIP = "/content/RoboVAD.zip"
if not os.path.exists("/content/RoboVAD/annotations.json"):
    total = int(urllib.request.urlopen(urllib.request.Request(URL, method="HEAD")).headers["Content-Length"])
    have = os.path.getsize(ZIP) if os.path.exists(ZIP) else 0
    if have < total:
        req = urllib.request.Request(URL, headers={"Range": f"bytes={have}-"})
        with urllib.request.urlopen(req) as r, open(ZIP, "ab") as f,                 tqdm(total=total, initial=have, unit="B", unit_scale=True, desc="下载") as bar:
            while chunk := r.read(1 << 20):
                f.write(chunk)
                bar.update(len(chunk))
    with zipfile.ZipFile(ZIP) as z:
        members = [m for m in z.infolist()
                   if m.filename.startswith("RoboVAD/") and not m.filename.endswith("_right.mp4")]
        for m in tqdm(members, desc="解压"):
            z.extract(m, "/content")
    os.remove(ZIP)
print(sorted(os.listdir("/content/RoboVAD")), len(os.listdir("/content/RoboVAD/videos")), "个视频")

In [ ]:
# 3. 缓存放 Google Drive（会弹授权窗口；不想挂就跳过这一格）
from google.colab import drive
drive.mount("/content/drive")
CACHE = "/content/drive/MyDrive/jianjin_cache"
os.makedirs(CACHE, exist_ok=True)
print("缓存目录", CACHE)

In [ ]:
# 4. 没挂 Drive 时的缓存位置
try:
    CACHE
except NameError:
    CACHE = "/content/cache"
    os.makedirs(CACHE, exist_ok=True)
print("缓存目录", CACHE)

In [ ]:
# 5a. 先下载 Robometer 权重（8.9 GB，带进度条）；下好了再运行这格会秒过
import psutil
from huggingface_hub import snapshot_download
def ram():
    m = psutil.virtual_memory()
    return f"内存 已用 {(m.total - m.available) / 1e9:.1f} / {m.total / 1e9:.1f} GB"
os.environ.pop("HF_HUB_DISABLE_XET", None)  # 下载用默认方式，才有进度条
print(snapshot_download("lerobot/Robometer-4B"))
print(ram())

In [ ]:
# 5b. 加载 Robometer（约 5 分钟）。这段时间显存一直是 0、内存在涨，是正常的：
# 先把权重读进内存、转成半精度，最后才搬到显卡。内存要是撑爆，会弹 session crashed。
import gc, time
gc.collect()
print("加载前", ram(), flush=True)
from jianjin.judges.robometer import RobometerJudge
J = RobometerJudge()
t = time.time()
J._load()
gc.collect()
print("加载完", f"{time.time() - t:.0f}s", J.version, ram())

In [ ]:
# 6. 冒烟：test 里前 3 条，打印曲线和耗时，估算全量时间
import csv, time, torch
from jianjin.ingest.robovad import read_robovad
from jianjin.judges.cache import judge_episode
ROOT = "/content/RoboVAD"
names = [r["episode_name"] for r in csv.DictReader(open(f"{ROOT}/splits/test.csv"))]
eps = read_robovad(ROOT, names=names[:3]).episodes
per = []
for ep in eps:
    torch.cuda.reset_peak_memory_stats()
    t = time.time()
    c = judge_episode(J, ep, "left", CACHE, sample_fps=2.0, max_side=336)
    per.append(time.time() - t)
    print(ep.id, f"{per[-1]:.1f}s", f"峰值 {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
    print("  进度", [round(p, 2) for p in c.progress])
    print("  标注", ep.meta["anomalies"])
print(f"全量 996 条预计 {sum(per) / len(per) * 996 / 3600:.1f} 小时")

In [ ]:
# 7. 全量：先 test 再 train；断了重跑这一格，已算完的直接跳过
import gc
from jianjin.checks import NotApplicable
for split in ["test", "train"]:
    names = [r["episode_name"] for r in csv.DictReader(open(f"{ROOT}/splits/{split}.csv"))]
    eps = read_robovad(ROOT, names=names).episodes
    t0 = time.time(); done = skipped = 0
    for i, ep in enumerate(eps, 1):
        try:
            judge_episode(J, ep, "left", CACHE, sample_fps=2.0, max_side=336)
            done += 1
        except NotApplicable as e:
            skipped += 1; print("跳过", ep.id, e)
        except torch.OutOfMemoryError:
            skipped += 1; print("显存不够，跳过", ep.id)
            gc.collect(); torch.cuda.empty_cache()
        if i % 20 == 0 or i == len(eps):
            print(f"{split} {i}/{len(eps)}，完成 {done}，跳过 {skipped}，用时 {(time.time() - t0) / 60:.0f} 分钟", flush=True)

In [ ]:
# 8. 打包下载
import shutil
from google.colab import files
shutil.make_archive("/content/robometer_cache", "zip", CACHE)
files.download("/content/robometer_cache.zip")